# Phase 2: Text Preprocessing & Class Imbalance Mitigation

**Objective:**
Transform the raw, noisy text into a clean, standardized format suitable for both Deep Learning (LSTM/Transformers) and Traditional Machine Learning models. This pipeline ensures data integrity by handling missing values, duplicates, and linguistic anomalies before applying balancing techniques.

**Workflow:**
1. Initial Data Health Check & Multi-label Aggregation (Merge raters' annotations)
2. Regex-based Text Purification (URLs, mentions, special characters)
3. Post-cleaning Quality Assurance (Remove empty strings & single-word outliers)
4. Data Splitting & Class Weight Calculation

## 1. Initial Data Health Inspection
Before executing any drop operations, it is critical to inspect the exact volume and nature of missing values (NaNs) and duplicated rows to prevent unintended data loss and detect potential scraping bugs.

In [11]:
# 0. ENVIRONMENT SETUP & IMPORTS
import pandas as pd
import numpy as np 
import re
import warnings  

# Suppress minor warnings for clean output
warnings.filterwarnings('ignore')

# Update path to the GoEmotions Parquet file
RAW_DATA_PATH = r"C:\nlp_project\data\raw\train-00000-of-00001.parquet"
PROCESSED_DATA_PATH = r"C:\nlp_project\data\processed"

# Redefine the 28 emotion columns to be used throughout the file
EMOTION_COLUMNS = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness",
    "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral"
]

print("Environment ready. Libraries imported.")

Environment ready. Libraries imported.


In [12]:
# 1. INITIAL DATA HEALTH INSPECTION

print("Loading raw GoEmotions Parquet dataset into memory...") 
df_raw = pd.read_parquet(RAW_DATA_PATH)
initial_shape = df_raw.shape

print("=== RAW DATA HEALTH REPORT ===")
print(f"Initial total rows: {df_raw.shape[0]}\n")

# 1.1 CHECK FOR MISSING VALUES (NaNs)
missing_counts = df_raw['text'].isnull().sum()
print("--- 1. Missing Values in Text ---")
if missing_counts > 0:
    print(f"Detected {missing_counts} missing texts.")
    print("\n[Sample of rows with Missing Texts]:")
    display(df_raw[df_raw['text'].isnull()].head(3))
else:
    print("-> Excellent: No missing text values detected!")

# 1.2 CHECK FOR DUPLICATED TEXTS
duplicate_count = df_raw.duplicated(subset=['text']).sum() 
print(f"\n--- 2. Duplicated Texts (Multi-Rater Annotations) ---") 
print(f"Detected {duplicate_count} duplicated text records.")
print("NOTE: In GoEmotions, these are NOT bugs! They represent different raters annotating the same sentence.")

# Inspect a sample of duplicated pairs to visually prove they have different labels
if duplicate_count > 0:
    print("\n[Sample: One sentence annotated by multiple raters]:")
    # Find one specific text that has duplicates
    sample_duplicate_text = df_raw[df_raw.duplicated(subset=['text'])]['text'].iloc[0]
    
    # Filter the dataframe to show all rows containing this exact text
    # Only display the text and the emotion columns to keep it clean
    cols_to_show = ['text'] + EMOTION_COLUMNS
    duplicates_sample = df_raw[df_raw['text'] == sample_duplicate_text][cols_to_show]
    
    # Display the rows where you will see the 1s in different emotion columns
    display(duplicates_sample) 

Loading raw GoEmotions Parquet dataset into memory...
=== RAW DATA HEALTH REPORT ===
Initial total rows: 211225

--- 1. Missing Values in Text ---
-> Excellent: No missing text values detected!

--- 2. Duplicated Texts (Multi-Rater Annotations) ---
Detected 153493 duplicated text records.
NOTE: In GoEmotions, these are NOT bugs! They represent different raters annotating the same sentence.

[Sample: One sentence annotated by multiple raters]:


,text,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,desire,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
103,"[NAME] to [NAME]: ""Don't try and tell me how t...",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
244,"[NAME] to [NAME]: ""Don't try and tell me how t...",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
187554,"[NAME] to [NAME]: ""Don't try and tell me how t...",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [13]:
# 1.3 MULTI-LABEL AGGREGATION (MERGING OVERLAPS)
print("\nExecuting Multi-Label Aggregation...")

# 1. Drop complete NaNs in text just in case to prevent groupby errors
df_clean = df_raw.dropna(subset=['text']).copy()

# 2. THE MULTI-LABEL MAGIC: Group by 'text' and take the maximum value for each emotion
temp_shape = df_clean.shape[0]

# Group by text and take the max value (1) across the 28 emotion columns
df_clean = df_clean.groupby('text')[EMOTION_COLUMNS].max().reset_index()

rows_merged = temp_shape - df_clean.shape[0]

print(f"Successfully aggregated {rows_merged} multi-rater overlapping rows into unique single rows.")
print(f"Dataset shape ready for Regex pipeline: {df_clean.shape}") 

# Display a proof of aggregation (a sentence that now holds multiple labels)
print("\n[Sample of Aggregated Multi-Label Row]:")
sample_multi = df_clean[df_clean[EMOTION_COLUMNS].sum(axis=1) > 2].head(2)
display(sample_multi) 


Executing Multi-Label Aggregation...
Successfully aggregated 153493 multi-rater overlapping rows into unique single rows.
Dataset shape ready for Regex pipeline: (57732, 29)

[Sample of Aggregated Multi-Label Row]:


,text,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,desire,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,"""If you don't wear BROWN AND ORANGE...YOU DON...",0,0,1,1,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,1
5,>sexuality shouldn’t be a grouping category I...,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,1


## 2. Deep Learning Surface Cleaning (Digital Noise & Normalization)
Before altering the text, we must inspect the dataset for digital artifacts (URLs, HTML tags, @mentions). For Deep Learning models (BERT/LSTM), we aim to remove these meaningless artifacts while preserving semantic structures (lowercase, expanded contractions, and expressive punctuations like `! ? . ,`).

### 2.1 Inspect Digital Noise Footprints

In [14]:
# 2.1 INSPECT DIGITAL NOISE (PRE-REGEX)
import re

print("Scanning dataset for digital noise and GoEmotions artifacts...\n") 

# Define basic patterns to detect common digital noise and specific tags
url_pattern = r'http[s]?://\S+|www\.\S+'
mention_pattern = r'@\w+'
html_pattern = r'&\w+;|<[^>]+>'
# Specifically look for GoEmotions anonymization tags (e.g., [NAME], [RELIGION])
goemotions_tag_pattern = r'\[[A-Z]+\]'

# Generate boolean masks to find rows containing these patterns
url_mask = df_clean['text'].str.contains(url_pattern, regex=True, na=False)
mention_mask = df_clean['text'].str.contains(mention_pattern, regex=True, na=False)
html_mask = df_clean['text'].str.contains(html_pattern, regex=True, na=False)
tag_mask = df_clean['text'].str.contains(goemotions_tag_pattern, regex=True, na=False)

print("=== NOISE FOOTPRINT REPORT ===")
print(f"Contains URLs         : {url_mask.sum()} rows")
print(f"Contains Mentions     : {mention_mask.sum()} rows")
print(f"Contains HTML tags    : {html_mask.sum()} rows")
print(f"Contains [TAGS]       : {tag_mask.sum()} rows")

# Inspect actual samples if noise is detected
if tag_mask.sum() > 0:
    print("\n[Sample] Texts with GoEmotions [TAGS]:")
    print(df_clean[tag_mask]['text'].head(3).to_string())
    
if url_mask.sum() > 0:
    print("\n[Sample] Texts with URLs:")
    print(df_clean[url_mask]['text'].head(2).to_string()) 

Scanning dataset for digital noise and GoEmotions artifacts...

=== NOISE FOOTPRINT REPORT ===
Contains URLs         : 0 rows
Contains Mentions     : 9 rows
Contains HTML tags    : 4 rows
Contains [TAGS]       : 8093 rows

[Sample] Texts with GoEmotions [TAGS]:
3      '*Pray*, v. To ask that the laws of the unive...
7                   Best number! [NAME], [NAME], [NAME]
20     No mention of [NAME]? For shame, subreddit. F...


### 2.2 Execute Deep Learning Text Purification
Based on the inspection, we apply a targeted regex engine. Unlike traditional ML preprocessing, we DO NOT remove stopwords or apply stemming. We preserve the natural grammar and structural punctuations necessary for Transformer self-attention mechanisms.

In [15]:
# 2.2 EXECUTE DL TEXT PURIFICATION 
import re
import contractions
from emot.core import emot
from tqdm import tqdm

# Enable pandas progress bar functionality
tqdm.pandas(desc="Purifying DL Text Data")

# Initialize the auto-emoticon translation engine 
emot_engine = emot()

def clean_text_dl(text: str) -> str:
    """
    Purifies text explicitly for Deep Learning models.
    Auto-translates ALL emoticons using 'emot' library, handles tags, 
    expands contractions, and applies strict regex filtering.
    """
    text = str(text)
    
    # 0. Handle GoEmotions specific tags before processing
    text = re.sub(r'\[NAME\]', 'someone', text)
    text = re.sub(r'\[RELIGION\]', 'religion', text)
    text = re.sub(r'\[[A-Z]+\]', ' ', text)
    
    # 1. EMOTION PRESERVATION (AUTOMATED): Detect and translate emoticons
    emoticon_info = emot_engine.emoticons(text)
    if emoticon_info and emoticon_info['flag']:
        # emoticon_info returns a dict with the symbol ('value') and its meaning ('mean')
        # Example: value=':)', mean='Happy face smiley'
        for value, mean in zip(emoticon_info['value'], emoticon_info['mean']):
            # Extract the meaning phrase (remove extraneous characters from the meaning string)
            clean_mean = re.sub(r'[^a-zA-Z\s]', '', mean).lower()
            # Replace the original symbol with its extracted text meaning
            text = text.replace(value, f" {clean_mean} ")
    
    # 2. Expand ALL English contractions automatically
    text = contractions.fix(text)
    
    # 3. Convert to lowercase 
    text = text.lower()
            
    # 4. Remove URLs, mentions, and HTML tags
    text = re.sub(r'http[s]?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'&\w+;|<[^>]+>', ' ', text)
    
    # 5. Strict Regex: Keep ONLY alphabets, spaces, and basic expressive punctuations
    text = re.sub(r'[^a-z\s\!\?\.\,]', ' ', text)
    
    # 6. Collapse multiple consecutive spaces into a single space and strip edges
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print(f"Applying Auto-Emoticon & Regex engine to {df_clean.shape[0]} rows...")

# Create a new column 'clean_text' to preserve the original for comparison
df_clean['clean_text'] = df_clean['text'].progress_apply(clean_text_dl)

print("\nDL Text purification completed.")  

Applying Auto-Emoticon & Regex engine to 57732 rows...


Purifying DL Text Data: 100%|██████████| 57732/57732 [00:00<00:00, 69410.37it/s]


DL Text purification completed.


## 3. Post-Cleaning Quality Assurance (QA)
The regex engine may have reduced some texts into empty strings (ghost texts) or left us with ultra-short sentences. For Deep Learning, sentences with fewer than 3 words often lack sufficient semantic context for the self-attention mechanism and can introduce label noise. We will inspect and filter these out.

### 3.1 Inspect Post-Cleaning Anomalies

In [16]:
# 3.1 INSPECT POST-CLEANING ANOMALIES
print("Scanning for post-cleaning anomalies...\n") 

# Calculate word count based on the newly cleaned text
df_clean['word_count_clean'] = df_clean['clean_text'].apply(lambda x: len(str(x).split()))

# Detect empty strings (ghost texts: word count is 0)
empty_mask = df_clean['word_count_clean'] == 0

# Detect ultra-short sentences (1 or 2 words)
# For GoEmotions, 2 words might actually hold meaning (e.g., "thank you"), 
# but let's stick to your strict threshold of requiring at least 3 words for better context.
short_mask = (df_clean['word_count_clean'] > 0) & (df_clean['word_count_clean'] < 3)

print("=== POST-CLEANING ANOMALY REPORT ===")
print(f"Empty strings (Ghost texts) : {empty_mask.sum()} rows")
print(f"Ultra-short sentences (< 3) : {short_mask.sum()} rows")

# Inspect samples of empty strings
if empty_mask.sum() > 0:
    print("\nGhost texts (became empty after regex):")
    # Removed the 'label' column from the print statement
    print(df_clean[empty_mask][['text', 'clean_text']].head(3).to_string())

# Inspect samples of ultra-short sentences
if short_mask.sum() > 0:
    print("\nUltra-short sentences:") 
    # Removed the 'label' column from the print statement
    print(df_clean[short_mask][['text', 'clean_text']].head(5).to_string())
else:
    print("\n-> No ultra-short sentences found!") 

Scanning for post-cleaning anomalies...

=== POST-CLEANING ANOMALY REPORT ===
Empty strings (Ghost texts) : 3 rows
Ultra-short sentences (< 3) : 1154 rows

Ghost texts (became empty after regex):
                           text clean_text
1115                        :^(           
1123   <<creepy loui face.jpg>>           
57710                  😘 ☂ ☂️☂️           

Ultra-short sentences:
                     text             clean_text
53  !Messageme creepypost  !messageme creepypost
54        !RemindMe 1 day          !remindme day
55      !RemindMe 10 days         !remindme days
56     !RemindMe 10 hours        !remindme hours
57     !RemindMe 24 hours        !remindme hours


### 3.2 Execute Quality Assurance Filtering

In [17]:
# 3.2 EXECUTE POST-CLEANING QA (FILTERING)
print("Executing Post-cleaning QA...")

initial_qa_shape = df_clean.shape[0]

# Filter out empty strings and short sentences (Keep only rows with >= 3 words)
df_clean = df_clean[df_clean['word_count_clean'] >= 3]

dropped_total = initial_qa_shape - df_clean.shape[0]

df_clean['text'] = df_clean['clean_text']
df_clean = df_clean.drop(columns=['clean_text', 'word_count_clean'])

print(f"Successfully purged {dropped_total} anomalous rows (empty or < 3 words).")
print(f"Final purified dataset shape ready for splitting: {df_clean.shape}") 

Executing Post-cleaning QA...
Successfully purged 1157 anomalous rows (empty or < 3 words).
Final purified dataset shape ready for splitting: (56575, 29)


## 4. Data Splitting & Class Imbalance Mitigation
Before partitioning the dataset, we must inspect the target label distribution to understand the severity of class imbalance. Following this, we will perform a stratified split (80% Train, 10% Validation, 10% Test) to preserve this exact distribution across all sets. Finally, we generate PyTorch class weights and an undersampled baseline set to assist the downstream Deep Learning models.

### 4.1 Inspect Label Distribution

In [18]:
# 4.1 INSPECT CLASS DISTRIBUTION (MULTI-LABEL)
print("Inspecting Target Label Distribution across 28 classes...\n") 

# Calculate absolute counts (sum of 1s) and percentages for each emotion column
label_counts = df_clean[EMOTION_COLUMNS].sum().sort_values(ascending=False)
label_percentages = (label_counts / len(df_clean)) * 100

print("=== CLASS DISTRIBUTION REPORT (DESCENDING) ===")
for label in label_counts.index:
    print(f"Label {label:>15}: {int(label_counts[label]):>6} positive rows ({label_percentages[label]:>5.2f}%)")

print("\n-> Note: Extreme imbalance detected in multi-label space (e.g., neutral vs grief).")
print("-> Pos_weights for BCEWithLogitsLoss are mandatory.") 

Inspecting Target Label Distribution across 28 classes...

=== CLASS DISTRIBUTION REPORT (DESCENDING) ===
Label         neutral:  30685 positive rows (54.24%)
Label        approval:  13094 positive rows (23.14%)
Label       annoyance:   9926 positive rows (17.54%)
Label      admiration:   9715 positive rows (17.17%)
Label     disapproval:   8356 positive rows (14.77%)
Label     realization:   7227 positive rows (12.77%)
Label  disappointment:   6614 positive rows (11.69%)
Label        optimism:   6185 positive rows (10.93%)
Label       curiosity:   6120 positive rows (10.82%)
Label             joy:   5605 positive rows ( 9.91%)
Label           anger:   5539 positive rows ( 9.79%)
Label       confusion:   5252 positive rows ( 9.28%)
Label       gratitude:   5132 positive rows ( 9.07%)
Label       amusement:   5091 positive rows ( 9.00%)
Label         sadness:   4626 positive rows ( 8.18%)
Label            love:   4307 positive rows ( 7.61%)
Label          caring:   4288 positive rows ( 

### 4.2 Execute Stratified Splitting & Export Artifacts

In [19]:
# 4.2 EXECUTE SPLIT & EXPORT ARTIFACTS
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

print("Initiating Data Splitting (80/10/10)...")

# Keep only the essential columns ('text' + 28 emotions)
columns_to_export = ['text'] + EMOTION_COLUMNS
df_final = df_clean[columns_to_export]

# Split 1: 80% Train, 20% Temp (for Val and Test)
# Note: stratify is removed because sklearn doesn't support 2D multi-label arrays natively.
# With ~56,000 rows, a random split ensures a statistically sound distribution.
train_df, temp_df = train_test_split(df_final, test_size=0.2, random_state=42)

# Split 2: Divide Temp equally into Val (10%) and Test (10%)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train Size: {train_df.shape[0]} rows")
print(f"Val Size  : {val_df.shape[0]} rows")
print(f"Test Size : {test_df.shape[0]} rows")

print("\nComputing Multi-Label Pos_Weights (PyTorch Tensor) from the Train set...")
total_train_samples = len(train_df)
pos_weights = []

for col in EMOTION_COLUMNS:
    num_positives = train_df[col].sum()
    num_negatives = total_train_samples - num_positives
    # Apply pos_weight formula: Negatives / Positives
    weight = num_negatives / (num_positives + 1e-5)
    pos_weights.append(weight)

# Convert list to PyTorch float32 tensor (Required for BCEWithLogitsLoss)
weights_tensor = torch.tensor(pos_weights, dtype=torch.float32)

# --- EXPORT ARTIFACTS ---
print("\nSaving all artifacts to disk...") 
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# Using os.path.join to prevent path concatenation errors
train_df.to_csv(os.path.join(PROCESSED_DATA_PATH, "train.csv"), index=False)
val_df.to_csv(os.path.join(PROCESSED_DATA_PATH, "val.csv"), index=False)
test_df.to_csv(os.path.join(PROCESSED_DATA_PATH, "test.csv"), index=False)

# Save PyTorch Tensor
torch.save(weights_tensor, os.path.join(PROCESSED_DATA_PATH, "pos_weights.pt"))

print(f"Pipeline complete! All artifacts exported successfully to: {PROCESSED_DATA_PATH}")
print("=== PREPROCESSING COMPLETED ===") 

Initiating Data Splitting (80/10/10)...
Train Size: 45260 rows
Val Size  : 5657 rows
Test Size : 5658 rows

Computing Multi-Label Pos_Weights (PyTorch Tensor) from the Train set...

Saving all artifacts to disk...
Pipeline complete! All artifacts exported successfully to: C:\nlp_project\data\processed
=== PREPROCESSING COMPLETED ===
